# Intent Classifier

Build a multi-class classifier using traditional ML on the labeled intent column. Categories should be condensed from 27 fine-grained intents into a manageable set for routing.

In [1]:
import sys
!{sys.executable} -m pip install datasets scikit-learn pandas numpy

## Load Dataset
Using `bitext/Bitext-customer-support-llm-chatbot-training-dataset`

In [2]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
df = pd.DataFrame(dataset['train'])
print(f'Total samples: {len(df)}')
df.head()

c:\Users\LEGION\anaconda3\envs\dental_xray_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\LEGION\anaconda3\envs\dental_xray_env\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LEGION\.cache\huggingface\hub\datasets--bitext--Bitext-customer-support-llm-chatbot-training-dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode 

Total samples: 26872


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


## Condense Intents

In [3]:
intent_mapping = {
    'track_order': 'order_status',
    'delivery_options': 'order_status',
    'delivery_period': 'order_status',
    'cancel_order': 'order_management',
    'change_order': 'order_management',
    'place_order': 'order_management',
    'check_invoice': 'billing_and_refunds',
    'get_refund': 'billing_and_refunds',
    'payment_issue': 'billing_and_refunds',
    'create_account': 'account_management',
    'edit_account': 'account_management',
    'delete_account': 'account_management',
    'switch_account': 'account_management',
    'recover_password': 'account_management',
    'complaint': 'complaint',
    'review': 'complaint',
}
df['condensed_intent'] = df['intent'].map(lambda x: intent_mapping.get(x, 'out_of_scope'))
print(df['condensed_intent'].value_counts())

condensed_intent
out_of_scope           10910
account_management      4987
billing_and_refunds     2996
order_management        2993
order_status            2989
complaint               1997
Name: count, dtype: int64


## Preprocessing & Vectorization

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['target'] = le.fit_transform(df['condensed_intent'])
X_train_texts, X_test_texts, y_train, y_test = train_test_split(df['instruction'], df['target'], test_size=0.2, random_state=42)
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_texts)
X_test = vectorizer.transform(X_test_texts)

## Model Training (Linear SVC)

In [5]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

model = LinearSVC(random_state=42, max_iter=2000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.996093023255814
                     precision    recall  f1-score   support

 account_management       1.00      1.00      1.00       956
billing_and_refunds       0.99      0.98      0.99       592
          complaint       1.00      1.00      1.00       427
   order_management       0.99      1.00      1.00       565
       order_status       1.00      0.99      1.00       587
       out_of_scope       1.00      1.00      1.00      2248

           accuracy                           1.00      5375
          macro avg       1.00      1.00      1.00      5375
       weighted avg       1.00      1.00      1.00      5375



## Save Model

In [6]:
import joblib
import os
os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/intent_model.pkl')
joblib.dump(vectorizer, 'models/intent_vectorizer.pkl')
joblib.dump(le, 'models/intent_encoder.pkl')
print('Model saved')

Model saved
